# Inicio

In [1]:
# Importación de librerías necesarias
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs

# Gráficos y Visualización
import matplotlib.pyplot as plt
from matplotlib import style
style.use('ggplot') or plt.style.use('ggplot')
import seaborn as sns
import missingno as msno

# Preprocesado, Reducción de Dimensionalidad y Modelado
from sklearn.cluster import KMeans
from sklearn.preprocessing import scale, StandardScaler
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

In [4]:
# Carga del dataset desde el almacenamiento local
url = 'segmentacion.csv'
data = pd.read_csv(url, delimiter=',')

In [5]:
data

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,10870,1967,Graduation,Married,61223.0,0,1,2013-06-13,46,709,...,5,0,0,0,0,0,0,3,11,0
2236,4001,1946,PhD,Together,64014.0,2,1,2014-06-10,56,406,...,7,0,0,0,1,0,0,3,11,0
2237,7270,1981,Graduation,Divorced,56981.0,0,0,2014-01-25,91,908,...,6,0,1,0,0,0,0,3,11,0
2238,8235,1956,Master,Together,69245.0,0,1,2014-01-24,8,428,...,3,0,0,0,0,0,0,3,11,0


# EDA

In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

In [7]:
data

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,10870,1967,Graduation,Married,61223.0,0,1,2013-06-13,46,709,...,5,0,0,0,0,0,0,3,11,0
2236,4001,1946,PhD,Together,64014.0,2,1,2014-06-10,56,406,...,7,0,0,0,1,0,0,3,11,0
2237,7270,1981,Graduation,Divorced,56981.0,0,0,2014-01-25,91,908,...,6,0,1,0,0,0,0,3,11,0
2238,8235,1956,Master,Together,69245.0,0,1,2014-01-24,8,428,...,3,0,0,0,0,0,0,3,11,0


In [8]:
# Conversión de la columna de fecha a formato datetime para cálculos de antigüedad
data['Dt_Customer'] = data['Dt_Customer'].astype('datetime64[ns]')

In [9]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   2240 non-null   int64         
 1   Year_Birth           2240 non-null   int64         
 2   Education            2240 non-null   str           
 3   Marital_Status       2240 non-null   str           
 4   Income               2216 non-null   float64       
 5   Kidhome              2240 non-null   int64         
 6   Teenhome             2240 non-null   int64         
 7   Dt_Customer          2240 non-null   datetime64[ns]
 8   Recency              2240 non-null   int64         
 9   MntWines             2240 non-null   int64         
 10  MntFruits            2240 non-null   int64         
 11  MntMeatProducts      2240 non-null   int64         
 12  MntFishProducts      2240 non-null   int64         
 13  MntSweetProducts     2240 non-null   int64  

In [ ]:
data.head(10)

In [ ]:
data.isnull()

In [ ]:
# Verificación de valores nulos por columna
display(data.isnull().sum())

In [ ]:
msno.matrix(data)
plt.show()

In [ ]:
data.describe().T

In [ ]:
# Histogramas de variables numéricas
data.hist(figsize=(50, 30), bins=30)
plt.show()

In [ ]:
data.select_dtypes(include=[np.number])



In [ ]:
# Mapa de calor de correlación

num_data = data.select_dtypes(include=[np.number])

plt.figure(figsize=(20, 8))
sns.heatmap(num_data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Matriz de Correlación")
plt.show()

In [ ]:
num_data.corr()

In [ ]:
# Boxplots para detectar outliers
plt.figure(figsize=(15, 6))
num_data.boxplot(rot=90)
plt.title("Boxplots de Variables Numéricas")
plt.show()



In [ ]:
for col in num_data.columns:
  plt.figure(figsize=(10, 2))
  sns.boxplot(x=num_data[col])
  plt.title(f"Boxplot de {col}")
  plt.show()

In [ ]:
(num_data["Income"]>130000).sum()

In [ ]:
# # Pairplot para relaciones entre variables
# sns.pairplot(data,hue='Education')
# plt.show()

In [ ]:
# # Pairplot para relaciones entre variables
# sns.pairplot(data,hue='Marital_Status', x_vars= ["Income","MntWines"], y_vars=["MntFishProducts","NumCatalogPurchases","Income"])
# plt.show()

In [ ]:
# Gráficos de barras para variables categóricas

# col = 'Education'
col = 'Marital_Status'

plt.figure(figsize=(10, 5))
sns.countplot(x=data[col],palette="bright")
plt.title(f"Distribución de {col}")
plt.show()

# }}}}


In [ ]:
data.tail(10)

In [ ]:
data.info()

In [ ]:
##Revisión del sesgo en las variables numéricas
num_data.skew(axis=0) # Calculate skewness only for numerical columns

In [ ]:
# Instalamos la librería (si no está)
!pip install ydata-profiling --quiet

import pandas as pd
from ydata_profiling import ProfileReport

# 1. Cargar dataset
df = pd.read_csv("/content/segmentacion.csv")

# 2. Generar reporte de exploración
profile = ProfileReport(
    df,
    title="Reporte Exploratorio - Segmentación",
    explorative=True
)

# 3. Guardar reporte en HTML
profile.to_file("/content/reporte_segmentacion.html")

print("✅ Reporte generado: /content/reporte_segmentacion.html")


In [ ]:
from google.colab import files
files.download("/content/reporte_segmentacion.html")

In [ ]:
data

# Preparación data Clusterización

In [ ]:
from datetime import date
# Cálculo de variables derivadas: Antigüedad del cliente y Edad
data['Hoy'] = date.today()
data['Hoy'] = data['Hoy'].astype('datetime64[ns]')
data['Antiguedad'] = (data['Hoy'] - data['Dt_Customer']).dt.days / 365.25
data['Antiguedad'] = data['Antiguedad'].astype('int64')
data['Edad'] = 2025 - data['Year_Birth']

# Eliminación de columnas originales tras la transformación
data = data.drop(columns=['Hoy', 'Dt_Customer', 'Year_Birth'])

In [ ]:
# data=data.drop(columns = ['Hoy'])

In [ ]:
len(data)

Imputación

In [ ]:
media_income = data["Income"].mean()

In [ ]:
media_income1 = data["Income"][(data["Income"]<100000)].mean()

In [ ]:
from scipy.stats import trim_mean

# Definir el porcentaje de datos a recortar (por ejemplo, 10% de cada extremo)
truncation_percentage = 0.1  # 10%

# Calcular la media truncada
media_truncada = trim_mean(data["Income"], proportiontocut=truncation_percentage)



In [ ]:
mediana_income = data["Income"].median()

In [ ]:
print('Media:',media_income)
print('Media <100.000 :',media_income1)
print('Media truncada 10%:',media_truncada)
print('Mediana:',mediana_income)


In [ ]:
# Imputación de valores nulos en 'Income' utilizando la mediana para evitar sesgos de outliers
data = data.replace("NaN", mediana_income)
data = data.replace(np.nan, mediana_income)

In [ ]:
# Filtrado de valores atípicos (Outliers) basado en el análisis visual previo
data = data[(data["Edad"] < 90)]
data = data[(data["Income"] < 140000)]
data = data[(data["MntWines"] < 1500)]
data = data[(data["MntMeatProducts"] < 1000)]
data = data[(data["MntGoldProds"] < 250)]
data = data[(data["NumDealsPurchases"] < 10)]
data = data[(data["NumWebPurchases"] < 15)]
data = data[(data["NumCatalogPurchases"] < 12)]

In [ ]:
len(data)

In [ ]:
id_ =data['ID']
id_ = pd.DataFrame(id_)

In [ ]:
id_

In [ ]:
#Eliminar variables
data_segm=data.drop(columns = ['Z_CostContact','Complain','Z_Revenue','Response','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5','AcceptedCmp1','AcceptedCmp2','ID','AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1','AcceptedCmp2', 'Complain', 'Response'])

In [ ]:
# data_segm = data_segm.drop(columns = ['Hoy'])

In [ ]:
# Transformación de variables categóricas a numéricas mediante One-Hot Encoding
data_segm = pd.get_dummies(data_segm, columns=data_segm.select_dtypes(exclude=['int64', 'float64']).columns, drop_first=True)

In [ ]:
data_segm.select_dtypes(exclude=['int64','float64']).columns

In [ ]:
data_segm

**Normalización**

In [ ]:
# Estandarización de las variables (media=0, desviación=1)
scaler = StandardScaler().fit(data_segm)
data_segm_scaled = pd.DataFrame(scaler.transform(data_segm), index=data_segm.index.values, columns=data_segm.columns.values)

In [ ]:
data_segm_scaled.info()

In [ ]:
data_segm_scaled

# Kmeas


In [ ]:
# Método elbow para identificar el número óptimo de clusters
# ==============================================================================
range_n_clusters = range(1, 15)
inertias = []

for n_clusters in range_n_clusters:
    modelo_kmeans = KMeans(
                        n_clusters   = n_clusters,
                        n_init       = 20,
                        random_state = 123
                    )
    modelo_kmeans.fit(data_segm_scaled)
    inertias.append(modelo_kmeans.inertia_)

fig, ax = plt.subplots(1, 1, figsize=(6, 3.84))
ax.plot(range_n_clusters, inertias, marker='o')
ax.set_title("Evolución de la varianza intra-cluster total")
ax.set_xlabel('Número clusters')
ax.set_ylabel('Intra-cluster (inertia)');

In [ ]:
# Método silhouette para identificar el número óptimo de clusters
# ==============================================================================
range_n_clusters = range(2, 15)
valores_medios_silhouette = []

for n_clusters in range_n_clusters:
    modelo_kmeans = KMeans(
                        n_clusters   = n_clusters,
                        n_init       = 20,
                        random_state = 123
                    )
    cluster_labels = modelo_kmeans.fit_predict(data_segm_scaled)
    silhouette_avg = silhouette_score(data_segm_scaled, cluster_labels)
    valores_medios_silhouette.append(silhouette_avg)

fig, ax = plt.subplots(1, 1, figsize=(6, 3.84))
ax.plot(range_n_clusters, valores_medios_silhouette, marker='o')
ax.set_title("Evolución de media de los índices silhouette")
ax.set_xlabel('Número clusters')
ax.set_ylabel('Media índices silhouette');

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

print('Elbow Method to determine the number of clusters to be formed:')
Elbow_M = KElbowVisualizer(KMeans(), k=10)
Elbow_M.fit(data_segm_scaled)
Elbow_M.show()

In [ ]:
modelo_kmeans = KMeans(n_clusters=3, n_init=20, random_state=1995)
modelo_kmeans.fit(data_segm_scaled)

In [ ]:
sns.countplot(x=modelo_kmeans.labels_, palette="bright")

In [ ]:
data_segm['k-means_stand']=modelo_kmeans.labels_

In [ ]:
data_segm.groupby('k-means_stand').mean().T

In [ ]:
data_segm_scaled_fin = data_segm_scaled

data_segm_scaled_fin['k-means_stand'] = modelo_kmeans.labels_
data_segm_scaled_fin = data_segm_scaled_fin.groupby('k-means_stand').mean().T


In [ ]:
plt.figure(figsize=(20, 10))
sns.heatmap(data_segm_scaled_fin, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)

plt.title('Heatmap de Variables Promedio por Clúster')
plt.xlabel('Clúster')
plt.ylabel('Variable')
plt.show()

In [ ]:
data_segm.groupby('k-means_stand').count()

In [ ]:
# Pairplot para relaciones entre variables
# sns.pairplot(data_segm,hue='k-means_stand')
# plt.show()


In [ ]:
plt.figure()
pl=sns.swarmplot(x=data_segm["k-means_stand"], y=data_segm["Income"], color= "#CBEDDD", alpha=0.5 )
pl=sns.boxenplot(x=data_segm["k-means_stand"], y=data_segm["Income"])
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_avg = silhouette_score(data_segm_scaled, modelo_kmeans.labels_)
print("Índice de Silueta:", silhouette_avg)

# Kmeas PCA


In [ ]:
# Reducción a 9 componentes principales para capturar la mayor varianza posible
pca = PCA(n_components=9)
pca.fit(data_segm_scaled)
PCA_ds = pd.DataFrame(pca.transform(data_segm_scaled), columns=([f"col{i}" for i in range(1, 10)]))

# Varianza explicada por cada componente
explained_variance = pca.explained_variance_ratio_

In [ ]:
plt.plot(np.cumsum(explained_variance))
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulada')
plt.title('Curva del Codo para PCA')
plt.grid()
plt.show()

In [ ]:
#A 3D Projection Of Data In The Reduced Dimension
x =PCA_ds["col1"]
y =PCA_ds["col2"]
z =PCA_ds["col3"]

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(x,y,z)
plt.show()

In [ ]:
# Método elbow para identificar el número óptimo de clusters
# ==============================================================================
range_n_clusters = range(1, 15)
inertias = []

for n_clusters in range_n_clusters:
    modelo_kmeans = KMeans(
                        n_clusters   = n_clusters,
                        n_init       = 20,
                        random_state = 123
                    )
    modelo_kmeans.fit(PCA_ds)
    inertias.append(modelo_kmeans.inertia_)

fig, ax = plt.subplots(1, 1, figsize=(6, 3.84))
ax.plot(range_n_clusters, inertias, marker='o')
ax.set_title("Evolución de la varianza intra-cluster total")
ax.set_xlabel('Número clusters')
ax.set_ylabel('Intra-cluster (inertia)');

In [ ]:
# Método silhouette para identificar el número óptimo de clusters
# ==============================================================================
range_n_clusters = range(2, 15)
valores_medios_silhouette = []

for n_clusters in range_n_clusters:
    modelo_kmeans = KMeans(
                        n_clusters   = n_clusters,
                        n_init       = 20,
                        random_state = 123
                    )
    cluster_labels = modelo_kmeans.fit_predict(PCA_ds)
    silhouette_avg = silhouette_score(PCA_ds, cluster_labels)
    valores_medios_silhouette.append(silhouette_avg)

fig, ax = plt.subplots(1, 1, figsize=(6, 3.84))
ax.plot(range_n_clusters, valores_medios_silhouette, marker='o')
ax.set_title("Evolución de media de los índices silhouette")
ax.set_xlabel('Número clusters')
ax.set_ylabel('Media índices silhouette');

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

print('Elbow Method to determine the number of clusters to be formed:')
Elbow_M = KElbowVisualizer(KMeans(), k=10)
Elbow_M.fit(PCA_ds)
Elbow_M.show()

In [ ]:
modelo_kmeans_pca = KMeans(n_clusters=6, n_init=20, random_state=1995)
modelo_kmeans_pca.fit(PCA_ds)

In [ ]:
sns.countplot(x=modelo_kmeans_pca.labels_, palette="bright")

In [ ]:
data_segm['k-means_PCA']=modelo_kmeans_pca.labels_

In [ ]:
data_segm.groupby('k-means_PCA').mean().T

In [ ]:
PCA_ds_fin = PCA_ds
PCA_ds_fin['k-means_PCA']=modelo_kmeans_pca.labels_

In [ ]:
PCA_ds_fin_cluster = PCA_ds_fin.groupby('k-means_PCA').mean().T

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(PCA_ds_fin_cluster, cmap='coolwarm', annot=True, fmt=".2f", linewidths=0.5)

plt.title('Heatmap de Variables Promedio por Clúster')
plt.xlabel('Clúster')
plt.ylabel('Variable')
plt.show()

In [ ]:
data_segm.groupby('k-means_PCA').count()

In [ ]:
# Pairplot para relaciones entre variables
sns.pairplot(data_segm,hue='k-means_PCA')
plt.show()


In [ ]:
plt.figure()
pl=sns.swarmplot(x=data_segm["k-means_PCA"], y=data_segm["Income"], color= "#CBEDDD", alpha=0.5 )
pl=sns.boxenplot(x=data_segm["k-means_PCA"], y=data_segm["Income"])
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_avg = silhouette_score(PCA_ds, modelo_kmeans_pca.labels_)
print("Índice de Silueta:", silhouette_avg)

#Dendrograma

In [ ]:
# Crear y visualizar el dendrograma

plt.figure(figsize=(10, 5))
sch.dendrogram(sch.linkage(data_segm_scaled, method='ward'))
plt.title('Dendrograma - Clustering Jerárquico')
plt.xlabel('Puntos de Datos')
plt.ylabel('Distancia Euclidiana')
plt.show()




In [ ]:
# Aplicar Clustering Jerárquico (seleccionar número de clusters)
n_clusters = 4  # Se puede ajustar según el dendrograma
hc = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = hc.fit_predict(data_segm_scaled)

In [ ]:
#  Evaluar con el Índice de Silueta
silhouette_avg = silhouette_score(data_segm_scaled, labels)
print(f'Índice de Silueta: {silhouette_avg:.3f}')

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

for k in range(2, 10):
    hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hc.fit_predict(data_segm_scaled)
    sil_score = silhouette_score(PCA_ds, labels)
    print(f'Clusters: {k}, Silueta: {sil_score:.3f}')


#Dendrograma PCA

In [ ]:
# Crear y visualizar el dendrograma
plt.figure(figsize=(10, 5))
sch.dendrogram(sch.linkage(PCA_ds, method='ward'))
plt.title('Dendrograma - Clustering Jerárquico')
plt.xlabel('Puntos de Datos')
plt.ylabel('Distancia Euclidiana')
plt.show()


In [ ]:
# Aplicar Clustering Jerárquico (seleccionar número de clusters)
n_clusters = 4  # Se puede ajustar según el dendrograma
hc = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
labels = hc.fit_predict(PCA_ds)

In [ ]:
#  Evaluar con el Índice de Silueta
silhouette_avg = silhouette_score(PCA_ds, labels)
print(f'Índice de Silueta: {silhouette_avg:.3f}')

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

for k in range(2, 10):
    hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hc.fit_predict(PCA_ds)
    sil_score = silhouette_score(PCA_ds, labels)
    print(f'Clusters: {k}, Silueta: {sil_score:.3f}')
